# Лабораторная работа №10

# "Реализация Transformer для классификации текста"

## 1. Выбор и описание датасета

Датасет IMDB содержит текстовые отзывы о фильмах, с метками positive (положительный) и negative (отрицательный). Задача — классификация сентимента текста (определить, положительный ли отзыв или отрицательный). Датасет состоит из 50 000 отзывов (25 000 для обучения и 25 000 для тестирования).

In [1]:
# import torch
# print(torch.__version__)
# print("CUDA:", torch.version.cuda)
# print("GPU available:", torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

## 2. Подготовка данных

1) Считываем датасет.

2) Преобразуем метки в числовые значения: 0 — отрицательный отзыв, 1 — положительный.

3) Разделяем данные на обучающий и тестовый наборы.

3) Токенизируем текст с помощью токенизатора BERT.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import torch
import torch.nn as nn
import torch.optim as optim
import math


# Загрузка датасета
df = pd.read_csv('IMDB_Dataset.csv')

# Преобразуем метки в числа
label_encoder = LabelEncoder()
df['sentiment'] = label_encoder.fit_transform(df['sentiment'])

# Разделяем данные на обучающую и тестовую выборки
train_texts, test_texts, train_labels, test_labels = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=42)

# Используем токенизатор BERT для преобразования текста в токены
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = self.texts.iloc[item]
        label = self.labels.iloc[item]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',  # исправлено
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True  # исправлено
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаем DataLoader для обучающей и тестовой выборки
train_dataset = IMDBDataset(train_texts, train_labels, tokenizer)
test_dataset = IMDBDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

C:\Users\admin\anaconda3\envs\torch111\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\admin\anaconda3\envs\torch111\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 3. Реализация модели Transformer

Теперь реализуем все необходимые компоненты:

- MultiHeadAttention: многоголовое внимание.

- PositionalEncoding: позиционное кодирование с использованием косинусных функций.

- TransformerEncoderLayer: один слой энкодера Transformer.

- TransformerEncoder: последовательность слоёв энкодера.

- TransformerClassifier: классификатор, использующий Transformer.

In [3]:
import torch.nn as nn
import math

# Многоголовое внимание (MultiHeadAttention)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

# Позиционное кодирование (cosine)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dim_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

# Transformer Encoder
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, dim_ff):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, dim_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Transformer Classifier
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, dim_ff, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional = PositionalEncoding(d_model)
        self.encoder = TransformerEncoder(num_layers, d_model, num_heads, dim_ff)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.positional(x)
        x = self.encoder(x)
        x = x.mean(dim=1)  # mean pooling
        return self.fc(x)

## 4. Обучение и оценка модели

Теперь создадим пайплайн обучения и оценивания модели, с учётом использования GPU.

In [4]:
# Проверяем доступность GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Перемещаем модель на GPU (если доступно)
model = TransformerClassifier(
    vocab_size=len(tokenizer.vocab),
    d_model=128,
    num_heads=8,
    num_layers=4,
    dim_ff=512,
    num_classes=2
).to(device)

# Оптимизатор и loss функция
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

Using device: cuda


In [5]:
# Функция для обучения модели на GPU
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        # Перемещаем данные на GPU
        input_ids = batch['input_ids'].squeeze(1).to(device)
        attention_mask = batch['attention_mask'].squeeze(1).to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    return total_loss / len(loader)

# Оценка модели
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            # Перемещаем данные на GPU
            input_ids = batch['input_ids'].squeeze(1).to(device)
            attention_mask = batch['attention_mask'].squeeze(1).to(device)
            labels = batch['labels'].to(device)

            preds = model(input_ids).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return correct / total


In [6]:
# Запуск обучения
epochs = 3
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    test_acc = evaluate(model, test_loader, device)
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Epoch 1/3 - Train Loss: 0.5493, Test Accuracy: 0.8186
Epoch 2/3 - Train Loss: 0.3617, Test Accuracy: 0.8403
Epoch 3/3 - Train Loss: 0.3057, Test Accuracy: 0.8615


## Заключение

В ходе лабораторной работы была реализована модель Transformer для классификации текста на базе датасета IMDB. Модель использует ключевые компоненты: MultiHeadAttention, PositionalEncoding, TransformerEncoderLayer и TransformerEncoder. Для преобразования текста в числовые представления использован BertTokenizer.

Модель была обучена с нуля, и результаты показали хорошую точность на тестовом наборе. Несмотря на высокую вычислительную сложность из-за длины последовательностей, удалось достичь удовлетворительных результатов. Работа также создала основу для расширения задачи в следующих лабораторных, например, с добавлением изображений для мультимодальной классификации.

Таким образом, работа успешно продемонстрировала реализацию и обучение модели Transformer для классификации текста.

Работа выполнена Гареевой Д.Р (507540), j4150